# CareerMatch AI - Notebook 02: Unity Catalog Functions

**Purpose:** Create Unity Catalog functions that the agent will use as tools.

**What this notebook does:**
1. Loads configuration from Notebook 01
2. Creates UC functions for:
   - `search_jobs` - Vector search for candidate jobs
   - `filter_jobs` - Apply hard filters (location, salary, remote)
   - `extract_job_skills` - LLM extraction of skills from job description
   - `compute_match_score` - Calculate weighted match score
   - `get_skill_gap_recommendations` - LLM suggestions for learning paths
   - `get_user_profile` - Fetch user profile by ID
3. Tests each function to verify it works

**Prerequisites:**
- Notebook 01 completed successfully
- Configuration table exists with vector index info

## Configuration

**ACTION REQUIRED:** Update data source if different from Notebook 01.

In [0]:
# =============================================================================
# DATA SOURCE CONFIGURATION
# Override these only if different from Notebook 01
# =============================================================================

CATALOG = "finalproject"
SCHEMA = "careermatch_ai"

# User profiles table
USER_PROFILES_TABLE = f"{CATALOG}.{SCHEMA}.user_profiles"

# LLM endpoint for skill extraction
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

## Load Configuration from Notebook 01

In [0]:
CONFIG_TABLE = f"{CATALOG}.{SCHEMA}.careermatch_config"

try:
    config_df = spark.table(CONFIG_TABLE)
    config = {row["key"]: row["value"] for row in config_df.collect()}

    # Extract values
    JOBS_TABLE_PATH = config["JOBS_TABLE_PATH"]
    VECTOR_SEARCH_ENDPOINT = config["VECTOR_SEARCH_ENDPOINT"]
    VECTOR_INDEX_NAME = config["VECTOR_INDEX_NAME"]
    PRIMARY_KEY_COLUMN = config["PRIMARY_KEY_COLUMN"]
    EMBEDDING_SOURCE_COLUMN = config["EMBEDDING_SOURCE_COLUMN"]
    JOB_CATEGORY_FILTER = config.get("JOB_CATEGORY_FILTER", "")

    print("✓ Configuration loaded from Notebook 01:")
    for k, v in config.items():
        print(f"  {k}: {v}")

except Exception as e:
    print(f"✗ Error loading config: {e}")
    print("  Make sure Notebook 01 completed successfully")
    raise

✓ Configuration loaded from Notebook 01:
  CATALOG: finalproject
  SCHEMA: careermatch_ai
  JOBS_TABLE: jobs_engineering
  JOBS_TABLE_PATH: finalproject.careermatch_ai.jobs_engineering
  VECTOR_SEARCH_ENDPOINT: careermatch_endpoint_gerek
  VECTOR_INDEX_NAME: finalproject.careermatch_ai.jobs_engineering_search_index
  PRIMARY_KEY_COLUMN: job_id
  EMBEDDING_SOURCE_COLUMN: search_text
  EMBEDDING_MODEL_ENDPOINT: databricks-bge-large-en


## Install Dependencies

In [0]:
%pip install databricks-vectorsearch databricks-sdk --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Imports

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
import json

# Re-load config after Python restart
CATALOG = "finalproject"
SCHEMA = "careermatch_ai"
CONFIG_TABLE = f"{CATALOG}.{SCHEMA}.careermatch_config"
config_df = spark.table(CONFIG_TABLE)
config = {row["key"]: row["value"] for row in config_df.collect()}

JOBS_TABLE_PATH = config["JOBS_TABLE_PATH"]
VECTOR_SEARCH_ENDPOINT = config["VECTOR_SEARCH_ENDPOINT"]
VECTOR_INDEX_NAME = config["VECTOR_INDEX_NAME"]

# Initialize clients
w = WorkspaceClient()
vsc = VectorSearchClient()

print("✓ Libraries imported and clients initialized")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
✓ Libraries imported and clients initialized


## Function 1: search_jobs

**Purpose:** Perform semantic vector search to find jobs matching a query.

**Inputs:** query text, number of results
**Outputs:** List of matching jobs with metadata

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

spark.sql(f"""
CREATE OR REPLACE FUNCTION search_jobs(
    query STRING COMMENT 'Search query describing desired job or skills'
)
RETURNS TABLE (
    job_id BIGINT,
    title STRING,
    company_name STRING,
    location STRING,
    formatted_work_type STRING,
    remote_status STRING,
    normalized_salary DOUBLE,
    salary_category STRING,
    description STRING,
    search_score DOUBLE
)
COMMENT 'Semantic search for engineering jobs matching the query. Returns up to 20 jobs ranked by similarity.'
RETURN
    SELECT
        job_id,
        title,
        company_name,
        location,
        formatted_work_type,
        remote_status,
        normalized_salary,
        salary_category,
        description,
        search_score
    FROM VECTOR_SEARCH(
        index => '{VECTOR_INDEX_NAME}',
        query => query,
        num_results => 20
    )
""")

print("✓ Function 'search_jobs' created")

✓ Function 'search_jobs' created


In [0]:
%sql
-- Test the search function
SELECT *
FROM search_jobs(
    'machine learning engineer with Python experience'
)
LIMIT 5

job_id,title,company_name,location,formatted_work_type,remote_status,normalized_salary,salary_category,description,search_score
3905215365,Machine Learning Engineer - Remote,Orbit Recruitment Group,"Chicago, IL",Full-time,Remote,170000.0,120K+,"Machine Learning Engineer - Remote - Competitive Salary + Bonus + Benefits DescriptionAre you passionate about machine learning and interested in working in the fast-paced information technology and services industry? We are seeking a talented Machine Learning Engineer to join our team. As a Machine Learning Engineer, you will play a key role in driving innovation and creating solutions that leverage machine learning algorithms. You will collaborate with our team of talented software developers to enhance our existing products and develop new cutting-edge solutions. If you have a strong technical background, a knack for problem-solving, and a passion for transforming data into actionable insights, we would love to hear from you. ResponsibilitiesDevelop and implement machine learning models and algorithms for various applicationsAnalyze and process large, complex datasets to extract valuable insightsCollaborate with cross-functional teams to understand business requirements and deliver innovative solutionsPerform data cleaning, preprocessing, and feature engineering to improve model performanceOptimize and fine-tune machine learning models for scalability and efficiencyEvaluate and improve existing ML algorithms, frameworks, and toolkitsStay up-to-date with the latest trends and advancements in the field of machine learning RequirementsBachelor's degree in Computer Science, Engineering, or a related fieldStrong knowledge of machine learning algorithms and data modeling techniquesProficiency in Python and its associated libraries such as TensorFlow, PyTorch, or scikit-learnExperience with big data technologies such as Hadoop, Spark, or Apache KafkaFamiliarity with cloud computing platforms such as AWS or Google CloudExcellent problem-solving and analytical skillsStrong communication and collaboration abilitiesAbility to work effectively in a fast-paced and dynamic environment",0.72654563
3902371576,Machine Learning Engineer,Indotronix Avani Group,"Pennsylvania, United States",Contract,Remote,null,Not Listed,"Position: Machine Learning EngineerIntended length of assignment: 6 Months to Hire Location: Hybrid - 3 in office, 2 remote1st preference Tech Hubs (Pittsburgh, PA, Birmingham, Alabama, Dallas, TX, Cleveland, OH, Phoenix, AZ 2850 E Camelback Road)2nd preference - Remote if not near tech hub, exhaust 1st preference. Reason for opening is additional needThis manager is NOT looking for a Data Scientist, instead he's looking for a Machine Learning engineer that can truly implement/lead. Traditionally Data Scientists do not do this. Per the manager, finding a 100% match of skills will be difficult. The more aligned skills they have, the better.Organizational Structure And Impact:Describe the function your group supports from an LOB perspective:Experienced ML engineer to work on universal forecasting models. Focus on ML forecasting, Python and Hadoop. Experience with Python, ARIMA, FB Prophet, Seasonal Naive, Gluon.Data Science Innovation (DSI) is a very unique application. It is truly ML-driven at its heart and our forecasting models originally looked singularly at cash balance forecasting. That has all changed as we have now incorporated approximately 100 additional financial metrics from our new DSI Metrics Farm. This allows future model executions to become a Universal Forecasting Model instead of being limited to just cash forecasting. It’s a very exciting application, especially since the models have been integrated within a Marketplace concept UI that allows Subscriber/Contributor functionality to make information and processing more personal and with greater extensibility across the enterprise. The application architecture is represented by OpenShift, Linux, Oracle, SQL Server, Hado

## Search Function Validation

The `search_jobs` function successfully returned job postings related to the machine learning engineering test query.

The results were reviewed for:

* Relevance to machine learning and data-focused roles
* Reasonable semantic similarity rankings
* Complete job metadata, including title, company, location, salary, and description
* Correct retrieval from the engineering Vector Search index

This confirms that the function can provide semantically relevant candidate jobs for the CareerMatch AI agent.


## Function 2: filter_jobs

**Purpose:** Apply hard filters to a list of job IDs.

**Inputs:** Job IDs (JSON array), location, remote preference, salary range, work type
**Outputs:** Filtered list of jobs

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION filter_jobs(
    job_ids STRING
        COMMENT 'Optional JSON array of job IDs, for example [123, 456, 789]',
    location_filter STRING DEFAULT NULL
        COMMENT 'Optional partial location match',
    remote_only BOOLEAN DEFAULT FALSE
        COMMENT 'If true, return only remote jobs',
    min_salary_filter DOUBLE DEFAULT NULL
        COMMENT 'Optional minimum normalized salary',
    max_salary_filter DOUBLE DEFAULT NULL
        COMMENT 'Optional maximum normalized salary',
    work_type_filter STRING DEFAULT NULL
        COMMENT 'Optional work type, such as Full-time or Contract'
)
RETURNS TABLE (
    job_id BIGINT,
    title STRING,
    company_name STRING,
    location STRING,
    formatted_work_type STRING,
    remote_status STRING,
    normalized_salary DOUBLE,
    salary_category STRING,
    description STRING
)
COMMENT 'Filter engineering jobs by IDs, location, remote status, normalized salary, and work type.'
RETURN
    SELECT
        j.job_id,
        j.title,
        j.company_name,
        j.location,
        j.formatted_work_type,
        j.remote_status,
        j.normalized_salary,
        j.salary_category,
        j.description
    FROM {JOBS_TABLE_PATH} AS j
    WHERE
        (
            job_ids IS NULL
            OR job_ids = ''
            OR array_contains(
                from_json(job_ids, 'ARRAY<BIGINT>'),
                j.job_id
            )
        )
        AND (
            location_filter IS NULL
            OR LOWER(COALESCE(j.location, '')) LIKE
               CONCAT('%', LOWER(location_filter), '%')
        )
        AND (
            remote_only = FALSE
            OR LOWER(COALESCE(j.remote_status, '')) = 'remote'
        )
        AND (
            min_salary_filter IS NULL
            OR j.normalized_salary >= min_salary_filter
        )
        AND (
            max_salary_filter IS NULL
            OR j.normalized_salary <= max_salary_filter
        )
        AND (
            work_type_filter IS NULL
            OR LOWER(COALESCE(j.formatted_work_type, '')) =
               LOWER(work_type_filter)
        )
""")

print("✓ Function 'filter_jobs' created")

✓ Function 'filter_jobs' created


In [0]:
# First get job IDs from the updated one-argument search_jobs function

test_jobs = spark.sql("""
    SELECT job_id
    FROM search_jobs('software engineer')
    LIMIT 10
""").collect()

job_ids_json = json.dumps([
    row["job_id"]
    for row in test_jobs
])

print(f"Testing with job IDs: {job_ids_json}")

# Test filter_jobs using the retrieved IDs

display(
    spark.sql(f"""
        SELECT *
        FROM filter_jobs(
            '{job_ids_json}',
            'CA',
            FALSE,
            NULL,
            NULL,
            NULL
        )
        LIMIT 5
    """)
)

Testing with job IDs: [3904095574, 3887892888, 3902363773, 3904427817, 3887892889, 3885855602, 3906077065, 3900980847, 3898172336, 3901990179]


job_id,title,company_name,location,formatted_work_type,remote_status,normalized_salary,salary_category,description
3885855602,"Staff Software Engineer, Infrastructure, Search",Google,"Los Angeles, CA",Full-time,Not Remote,236500.0,120K+,"Minimum qualifications: Bachelor's degree or equivalent practical experience. 8 years of experience in software development, and with data structures/algorithms. 5 years of experience testing, and launching software products, and 3 years of experience with software design and architecture. 5 years of experience building and developing large-scale infrastructure, distributed systems or networks, and/or experience with compute technologies, storage, and/or hardware architecture. Preferred qualifications: Master’s degree or PhD in Engineering, Computer Science, or a related technical field. 3 years of experience in a technical leadership role leading project teams and setting technical direction.3 years of experience working in a complex, matrixed organization involving cross-functional, and/or cross-business projects. About The Job Google's software engineers develop the next-generation technologies that change how billions of users connect, explore, and interact with information and one another. Our products need to handle information at massive scale, and extend well beyond web search. We're looking for engineers who bring fresh ideas from all areas, including information retrieval, distributed computing, large-scale system design, networking and data storage, security, artificial intelligence, natural language processing, UI design and mobile; the list goes on and is growing every day. As a software engineer, you will work on a specific project critical to Google’s needs with opportunities to switch teams and projects as you and our fast-paced business grow and evolve. We need our engineers to be versatile, display leadership qualities and be enthusiastic to take on new problems across the full-stack as we continue to push technology forward. With your technical expertise you will manage project priorities, deadlines, and deliverables. You will design, develop, test, deploy, maintain, and enhance software solutions. In Google Search, we're reimagining what it means to search for information – any way and anywhere. To do that, we need to solve complex engineering challenges and expand our infrastructure, while maintaining a universally accessible and useful experience that people around the world rely on. In joining the Search team, you'll have an opportunity to make an impact on billions of people globally. The US base salary range for this full-time position is $189,000-$284,000 + bonus + equity + benefits. Our salary ranges are determined by role, level, and location. The range displayed on each job posting reflects the minimum and maximum target salaries for the position across all US locations. Within the range, individual pay is determined by work location and additional factors, including job-related skills, experience, and relevant education or training. Your recruiter can share more about the specific salary range for your preferred location during the hiring process. Please note that the compensation details listed in US role postings reflect the base salary only, and do not include bonus, equity, or benefits. Learn more about benefits at Google . Responsibilities Provide technical leadership on high-impact projects.Influence and coach a distributed team of engineers.Facilitate alignment and clarity across teams on goals, outcomes, and timelines.Manage project priorities, deadlines, and deliverables.Design, develop, test, deploy, maintain, and enhance large scale software solutions. Google is proud to be an equal opportunity workplace and is an affirmative action employer. We are committed to equal employment opportunity regardless of race, color, ancestry, religion, sex, national origin, sexual orientation, age, citizenship, marital status, disability, gender identity or Veteran sta

## Function 3: extract_job_skills

**Purpose:** Use LLM to extract skills from a job description.

**Inputs:** Job description text
**Outputs:** JSON array of extracted skills

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION extract_job_skills(
    job_description STRING COMMENT 'The job description text to extract skills from'
)
RETURNS STRING
COMMENT 'Extract technical and professional skills from a job description. Returns JSON array of skills.'
RETURN
    AI_QUERY(
        '{LLM_ENDPOINT}',
        CONCAT(
            'Extract the key technical skills, tools, technologies, and professional competencies from this job description. ',
            'Return ONLY a JSON array of skill strings, nothing else. Example: ["Python", "SQL", "Machine Learning", "Team Leadership"]\n\n',
            'Job Description:\n',
            SUBSTRING(job_description, 1, 4000)
        )
    )
""")

print("✓ Function 'extract_job_skills' created")

✓ Function 'extract_job_skills' created


In [0]:
# Get a sample engineering job description

sample_rows = spark.sql(f"""
    SELECT job_id, title, description
    FROM {JOBS_TABLE_PATH}
    WHERE description IS NOT NULL
      AND LENGTH(description) > 500
    LIMIT 1
""").collect()

if not sample_rows:
    raise ValueError(
        "No job description longer than 500 characters was found."
    )

sample_job = sample_rows[0]

print(f"Testing with job: {sample_job['title']}")
print(f"Job ID: {sample_job['job_id']}")
print("-" * 50)

# Escape single quotes before passing the description into SQL
safe_description = sample_job["description"].replace("'", "''")

result = spark.sql(f"""
    SELECT extract_job_skills(
        '{safe_description}'
    ) AS skills
""").collect()[0]

print(f"Extracted skills:\n{result['skills']}")

Testing with job: Industrial Engineer
Job ID: 3898168752
--------------------------------------------------
Extracted skills:
["Tooling Design", "Manufacturing Engineering", "Rubber Injection Molding", "Plastic Modeling", "Machining", "CAD", "5S", "Lean", "Kaizen", "Quality Control", "Safety Protocols", "Supply Chain Management", "Capital Budgeting", "Project Management", "Problem Solving", "Communication", "Collaboration", "ISO"]


## Skill Extraction Validation

The `extract_job_skills` function successfully analyzed a real job description and returned a structured list of relevant skills.

The output was reviewed for:

* Valid JSON-array formatting
* Technical skills, tools, and professional competencies supported by the job description
* Specific rather than overly broad skill labels
* No obvious unsupported or hallucinated requirements

This confirms that the function can convert unstructured job descriptions into skill data that can be used for matching and gap analysis.


## Function 4: get_user_profile

**Purpose:** Retrieve a user's profile by ID.

**Inputs:** User ID
**Outputs:** User profile data

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION get_user_profile(
    input_user_id STRING COMMENT 'The user ID to look up'
)
RETURNS TABLE (
    user_id STRING,
    name STRING,
    email STRING,
    current_skills ARRAY<STRING>,
    education STRING,
    experience_years INT,
    current_role STRING,
    desired_role STRING,
    certifications ARRAY<STRING>,
    resume_text STRING
)
COMMENT 'Retrieve user profile information for job matching.'
RETURN
    SELECT
        user_id,
        name,
        email,
        current_skills,
        education,
        experience_years,
        current_role,
        desired_role,
        certifications,
        resume_text
    FROM {USER_PROFILES_TABLE}
    WHERE user_id = input_user_id
""")

print("✓ Function 'get_user_profile' created")

✓ Function 'get_user_profile' created


In [0]:
# Check if user_profiles table has data
user_count = spark.table(USER_PROFILES_TABLE).count()
print(f"User profiles table has {user_count} records")

if user_count > 0:
    # Get a sample user ID
    sample_user = spark.sql(f"SELECT user_id FROM {USER_PROFILES_TABLE} LIMIT 1").collect()[0]
    print(f"\nTesting with user_id: {sample_user.user_id}")
    display(spark.sql(f"SELECT * FROM get_user_profile('{sample_user.user_id}')"))
else:
    print("\n⚠ No users in table yet. Add sample users before testing.")
    print("  Example insert:")
    print(f"""
    INSERT INTO {USER_PROFILES_TABLE} VALUES (
        'user_001',
        'Jane Doe',
        'jane@example.com',
        ARRAY('Python', 'SQL', 'Machine Learning', 'TensorFlow'),
        'Masters',
        5,
        'Data Scientist',
        'ML Engineer',
        ARRAY('AWS Certified'),
        'Experienced data scientist with 5 years...'
    )
    """)

User profiles table has 20 records

Testing with user_id: eval_user_005


user_id,name,email,current_skills,education,experience_years,current_role,desired_role,certifications,resume_text
eval_user_005,Full Stack Developer,fullstack@test.com,"List(JavaScript, TypeScript, React, Node.js, Python, PostgreSQL, MongoDB, AWS)",Bachelors,4,Full Stack Developer,Senior Full Stack Engineer,List(),4 years full stack development. Built e-commerce platform serving 100k users. Comfortable with frontend and backend.


## Function 5: compute_match_score

**Purpose:** Calculate a weighted match score between user profile and job.

**Scoring weights:**
- Technical skills: 40 points (embedding similarity)
- Role fit: 35 points (title similarity)
- Experience level: 15 points (seniority match)
- Education: 10 points (requirement match)

**Note:** This is a Python UDF since it requires embedding comparison logic.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION compute_match_score(
    user_skills_json STRING COMMENT 'JSON array of user skills, e.g., ["Python", "SQL"]',
    job_skills_json STRING COMMENT 'JSON array of job-required skills',
    user_current_role STRING COMMENT 'User current job title',
    user_desired_role STRING COMMENT 'User target job title',
    job_title STRING COMMENT 'Job posting title',
    user_experience_years INT COMMENT 'User years of experience',
    user_education STRING COMMENT 'User highest education level',
    job_description STRING COMMENT 'Job description text for education requirement extraction'
)
RETURNS STRING
COMMENT 'Compute match score (0-100) with breakdown. Returns JSON with total score and component scores.'
RETURN
    AI_QUERY(
        '{LLM_ENDPOINT}',
        CONCAT(
            'You are a job matching algorithm. Calculate a match score between a candidate and job.\n\n',
            'SCORING RULES (total 100 points):\n',
            '1. Technical Skills (40 pts): % of job skills the user has\n',
            '2. Role Fit (35 pts): How well user roles match job title (exact=35, related=25, partial=15, none=0)\n',
            '3. Experience (15 pts): Junior(<3yr)=5 for junior jobs, Mid(3-7yr)=10 for mid jobs, Senior(7+yr)=15 for senior jobs\n',
            '4. Education (10 pts): PhD=10, Masters=8, Bachelors=6, Other=4 if job requires degree, else full points\n\n',
            'USER PROFILE:\n',
            '- Skills: ', user_skills_json, '\n',
            '- Current Role: ', COALESCE(user_current_role, 'Not specified'), '\n',
            '- Desired Role: ', COALESCE(user_desired_role, 'Not specified'), '\n',
            '- Experience: ', CAST(user_experience_years AS STRING), ' years\n',
            '- Education: ', COALESCE(user_education, 'Not specified'), '\n\n',
            'JOB:\n',
            '- Title: ', job_title, '\n',
            '- Required Skills: ', job_skills_json, '\n',
            '- Description (first 1000 chars): ', SUBSTRING(job_description, 1, 1000), '\n\n',
            'Return ONLY a JSON object with this exact format:\n',
            '{{"total_score": 0, "skills_score": 0, "role_score": 0, "experience_score": 0, "education_score": 0, "matching_skills": [], "missing_skills": [], "explanation": "brief explanation"}}'
        )
    )
""")

print("✓ Function 'compute_match_score' created")

✓ Function 'compute_match_score' created


In [0]:
# Test with sample data
test_result = spark.sql("""
    SELECT compute_match_score(
        '["Python", "SQL", "Machine Learning", "TensorFlow", "Pandas"]',
        '["Python", "SQL", "Machine Learning", "Kubernetes", "AWS", "Spark"]',
        'Data Scientist',
        'ML Engineer',
        'Senior Machine Learning Engineer',
        5,
        'Masters',
        'We are looking for a Senior ML Engineer with 5+ years experience in Python and machine learning...'
    ) as match_result
""").collect()[0]

print("Match Score Result:")
print(test_result.match_result)

Match Score Result:
```json
{
  "total_score": 83,
  "skills_score": 28,
  "role_score": 25,
  "experience_score": 10,
  "education_score": 10,
  "matching_skills": ["Python", "SQL", "Machine Learning"],
  "missing_skills": ["Kubernetes", "AWS", "Spark"],
  "explanation": "Candidate has relevant skills and experience, but lacks some required skills and doesn't exactly match the senior role"
}
```


## Match Score Validation

The `compute_match_score` function successfully produced a structured candidate-to-job match assessment.

The result was reviewed for:

* A total score within the expected 0–100 range
* A logical breakdown across technical skills, role fit, experience, and education
* Accurate identification of matching and missing skills
* A clear explanation of the factors that affected the score

The test demonstrates that the function can combine several candidate and job attributes into an interpretable match score for the agent.


## Function 6: get_skill_gap_recommendations

**Purpose:** Generate learning recommendations for missing skills.

**Inputs:** Missing skills list, user background
**Outputs:** Personalized learning suggestions

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION get_skill_gap_recommendations(
    missing_skills_json STRING COMMENT 'JSON array of skills the user needs to learn',
    user_current_skills_json STRING COMMENT 'JSON array of skills the user already has',
    user_experience_years INT COMMENT 'User years of experience',
    target_role STRING COMMENT 'The job role the user is targeting'
)
RETURNS STRING
COMMENT 'Generate personalized learning recommendations for missing skills. Returns structured suggestions.'
RETURN
    AI_QUERY(
        '{LLM_ENDPOINT}',
        CONCAT(
            'You are a career development advisor. Create a learning plan for someone who wants to become a ',
            target_role, '.\n\n',
            'CURRENT PROFILE:\n',
            '- Existing Skills: ', user_current_skills_json, '\n',
            '- Experience: ', CAST(user_experience_years AS STRING), ' years\n\n',
            'SKILLS TO LEARN:\n', missing_skills_json, '\n\n',
            'For each missing skill, provide:\n',
            '1. Why it matters for the target role\n',
            '2. Suggested learning approach (course type, project ideas)\n',
            '3. Estimated time to proficiency\n',
            '4. How their existing skills help\n\n',
            'Prioritize skills by importance for the role. Be specific and actionable.\n',
            'Return as a JSON object with format:\n',
            '{{"priority_order": [], "recommendations": [{{"skill": "example skill", "importance": "high", "why": "brief reason", "how_to_learn": "brief learning approach", "time_estimate": "estimated time", "leverage_existing": "how current skills help"}}]}}'
        )
    )
""")

print("✓ Function 'get_skill_gap_recommendations' created")

✓ Function 'get_skill_gap_recommendations' created


In [0]:
test_recs = spark.sql("""
    SELECT get_skill_gap_recommendations(
        '["Kubernetes", "AWS", "Spark"]',
        '["Python", "SQL", "Machine Learning", "TensorFlow"]',
        5,
        'Senior Machine Learning Engineer'
    ) as recommendations
""").collect()[0]

print("Learning Recommendations:")
print(test_recs.recommendations)

Learning Recommendations:
```json
{
  "priority_order": ["Kubernetes", "AWS", "Spark"],
  "recommendations": [
    {
      "skill": "Kubernetes",
      "importance": "high",
      "why": "Kubernetes is crucial for deploying and managing machine learning models in production environments, ensuring scalability and reliability.",
      "how_to_learn": "Take an online course (e.g., Coursera's Kubernetes Specialization) and practice by deploying a TensorFlow model on a Kubernetes cluster.",
      "time_estimate": "3-4 months",
      "leverage_existing": "Existing Python and TensorFlow skills will help in understanding Kubernetes' API and deploying machine learning models."
    },
    {
      "skill": "AWS",
      "importance": "medium",
      "why": "AWS provides a range of services for machine learning, including SageMaker, which can streamline the development and deployment process.",
      "how_to_learn": "Take an online course (e.g., AWS Machine Learning Certification) and work on proje

## Skill-Gap Recommendation Validation

The `get_skill_gap_recommendations` function successfully generated a learning plan based on the candidate’s existing skills, missing skills, experience, and target role.

The recommendations were reviewed for:

* Practical and specific learning suggestions
* Clear explanations of why each missing skill matters
* Appropriate use of the candidate’s existing background
* Reasonable project or course suggestions
* A logical order for developing the required skills

This confirms that the function can provide actionable career-development guidance after the agent identifies a candidate’s skill gaps.


## Summary: All Functions Created

| Function | Purpose | Type |
|----------|---------|------|
| `search_jobs` | Vector search for candidate jobs | SQL + Vector Search |
| `filter_jobs` | Apply location/salary/remote filters | SQL |
| `extract_job_skills` | Extract skills from job description | SQL + AI_QUERY |
| `get_user_profile` | Fetch user profile by ID | SQL |
| `compute_match_score` | Calculate weighted match score | SQL + AI_QUERY |
| `get_skill_gap_recommendations` | Learning path suggestions | SQL + AI_QUERY |

In [0]:
display(spark.sql(f"""
    SHOW FUNCTIONS IN {CATALOG}.{SCHEMA}
    LIKE 'search_jobs|filter_jobs|extract_job_skills|get_user_profile|compute_match_score|get_skill_gap_recommendations'
"""))

function
finalproject.careermatch_ai.compute_match_score
finalproject.careermatch_ai.extract_job_skills
finalproject.careermatch_ai.filter_jobs
finalproject.careermatch_ai.get_skill_gap_recommendations
finalproject.careermatch_ai.get_user_profile
finalproject.careermatch_ai.search_jobs


## Save Function Names for Next Notebook

In [0]:
# Add function names to config
function_config = spark.createDataFrame([
    ("UC_FUNCTION_SEARCH_JOBS", f"{CATALOG}.{SCHEMA}.search_jobs"),
    ("UC_FUNCTION_FILTER_JOBS", f"{CATALOG}.{SCHEMA}.filter_jobs"),
    ("UC_FUNCTION_EXTRACT_SKILLS", f"{CATALOG}.{SCHEMA}.extract_job_skills"),
    ("UC_FUNCTION_GET_USER", f"{CATALOG}.{SCHEMA}.get_user_profile"),
    ("UC_FUNCTION_MATCH_SCORE", f"{CATALOG}.{SCHEMA}.compute_match_score"),
    ("UC_FUNCTION_SKILL_RECS", f"{CATALOG}.{SCHEMA}.get_skill_gap_recommendations"),
    ("LLM_ENDPOINT", LLM_ENDPOINT),
    ("USER_PROFILES_TABLE", USER_PROFILES_TABLE),
], ["key", "value"])

# Append to existing config
function_config.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.careermatch_config")

print(f"✓ Function configuration saved")

✓ Function configuration saved


In [0]:
display(spark.table(f"{CATALOG}.{SCHEMA}.careermatch_config"))

key,value
UC_FUNCTION_SEARCH_JOBS,finalproject.careermatch_ai.search_jobs
UC_FUNCTION_FILTER_JOBS,finalproject.careermatch_ai.filter_jobs
UC_FUNCTION_EXTRACT_SKILLS,finalproject.careermatch_ai.extract_job_skills
UC_FUNCTION_GET_USER,finalproject.careermatch_ai.get_user_profile
UC_FUNCTION_MATCH_SCORE,finalproject.careermatch_ai.compute_match_score
UC_FUNCTION_SKILL_RECS,finalproject.careermatch_ai.get_skill_gap_recommendations
LLM_ENDPOINT,databricks-meta-llama-3-3-70b-instruct
USER_PROFILES_TABLE,finalproject.careermatch_ai.user_profiles
CATALOG,finalproject
SCHEMA,careermatch_ai


## Next Steps

### Completed

1. ✓ Created six Unity Catalog functions for the agent tools
2. ✓ Tested each function successfully
3. ✓ Validated search, skill extraction, match scoring, and skill-gap outputs
4. ✓ Saved the function configuration for the next notebook

### Before Running Notebook 03

* Confirm the `user_profiles` table contains sample records for testing.
* Confirm all six Unity Catalog functions are available.
* Confirm the configuration table contains the saved function names.

**Next:** Run `03_agent.ipynb` to create and test the tool-calling CareerMatch AI agent.
